## Интерполяция улыбок ##

### Цель построения поверхности

На первом этапе для каждого фиксированного срока $T_i$ строим интерполянт call-цены по страйку. Для отсутствия статического арбитража call-цена должна быть не возрастающей и выпуклой функцией страйка, а её производная должна удовлетворять условию $-D(T_i)\leq \frac{\partial C}{\partial K}\leq 0$.

После построения отдельных улыбок выбираем общую область по *forward moneyness* и строим единую ценовую поверхность.

Линейную интерполяцию нормированных call-цен по времени используем как простой безарбитражный ценовой *baseline*. Для вычисления *local volatility* отдельно строим гладкую поверхность полной подразумеваемой дисперсии $w(T,k)=T\sigma_{\mathrm{impl}}^2(T,k)$.

In [1]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

from src.black import implied_vol
from src.interpolation import RationalCubicSplineInterpolator, slopes_estimation, r_boundary
from src.arbitrage import surface_arbitrage_report
from src.plotting import dark_3d_scatter, dark_3d_surface

In [2]:
surface = pd.read_parquet(PROCESSED_DIR / 'surface.parquet')
surface

,expiration,symbol,last,bid,ask,volume,iv,delta,gamma,open_interest,...,option_kind,log_moneyness,total_variance,call_midf,midf,call_bid,call_ask,call_mid,impl_vol_market,total_variance_market
0,2026-11-05,SPXW261105P07010000,0.20,0.15,0.2,60,0.5752,-0.0024,0.0000,94,...,Put,-0.092453,0.001079,679.099706,0.174999,679.074708,679.124708,679.099708,0.627654,0.001079
1,2026-11-05,SPXW261105P07020000,0.20,0.15,0.2,59,0.5671,-0.0025,0.0000,22,...,Put,-0.091027,0.001049,669.100806,0.174999,669.075807,669.125807,669.100807,0.618852,0.001049
2,2026-11-05,SPXW261105P07025000,0.15,0.15,0.2,18,0.5630,-0.0026,0.0000,447,...,Put,-0.090315,0.001034,664.101356,0.174999,664.076356,664.126356,664.101356,0.614452,0.001034
3,2026-11-05,SPXW261105P07030000,0.15,0.15,0.2,7,0.5590,-0.0026,0.0000,37,...,Put,-0.089604,0.001020,659.101906,0.175000,659.076906,659.126906,659.101906,0.610052,0.001020
4,2026-11-05,SPXW261105P07040000,0.15,0.15,0.2,14,0.5509,-0.0027,0.0000,15,...,Put,-0.088182,0.000990,649.103008,0.175003,649.078005,649.128005,649.103005,0.601255,0.000990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8582,2027-09-30,SPXW270930C09490000,0.00,34.40,35.4,0,0.1290,0.0870,0.0002,0,...,Call,0.175303,0.014982,34.859675,34.859675,34.400000,35.400000,34.900000,0.128762,0.014990
8583,2027-09-30,SPXW270930C09495000,0.00,34.00,35.0,0,0.1290,0.0863,0.0002,0,...,Call,0.175830,0.014978,34.520165,34.520165,34.000000,35.000000,34.500000,0.128695,0.014974
8584,2027-09-30,SPXW270930C09500000,27.40,33.70,34.7,0,0.1289,0.0856,0.0002,102,...,Call,0.176357,0.014979,34.199998,34.199998,33.700000,34.700000,34.200000,0.128713,0.014979
8585,2027-09-30,SPXW270930C09600000,36.12,27.40,28.2,0,0.1284,0.0723,0.0001,25,...,Call,0.186828,0.014857,27.800000,27.800000,27.400000,28.200000,27.800000,0.128189,0.014857


### Рациональный кубический сплайн Дельборго–Грегори ###

*Рациональный кубический сплайн* интерполирует точки $(x_i, f_i)$ на каждом интервале $[x_i, x_{i+1}]$ рациональной функцией, задаваемой значениями $\{f_i, f_{i+1}\}$ и наклонами $\{d_i, d_{i+1}\}$:
$$s(x) = \frac{P_i(\theta)}{Q_i(\theta)}, \qquad \theta = \frac{x - x_i}{x_{i+1} - x_i},$$
$$P_i(\theta) = f_{i+1}\theta^3 + (r_i f_{i+1} - h_i d_{i+1})\theta^2(1-\theta) + (r_i f_i + h_i d_i)\theta(1-\theta)^2 + f_i(1-\theta)^3,$$
$$Q_i(\theta) = 1 + (r_i - 3)\theta(1-\theta),$$
где $h_i = x_{i+1} - x_i$, а $r_i > -1$ — *параметр кубики*.

**Выбор $r_i$.** При условии знакопостоянной монотонности
$$0 \leq \pm d_1 < \pm \Delta_1 < \pm d_2 < \dots < \pm d_i < \pm \Delta_i < \dots < \pm d_n, \qquad \Delta_i = \frac{f_{i+1} - f_i}{x_{i+1} - x_i},$$
сплайн выпукл и монотонен, если
$$r_i \geq \max\left\{\frac{d_{i+1} - d_i}{d_{i+1} - \Delta_i},\ \frac{d_{i+1} - d_i}{\Delta_i - d_i}\right\}.$$

**Аппроксимация наклонов $d_i$** (геометрическое среднее, порядок $O(h^2)$):
$$d_i = \pm(\pm\Delta_{i-1})^{(x_{i+1} - x_i)/(x_{i+1} - x_{i-1})}(\pm\Delta_i)^{(x_i - x_{i-1})/(x_{i+1} - x_{i-1})}, \quad i = 2, \dots, n-1,$$
с концевыми условиями
$$d_1 = \pm(\pm\Delta_1)^{(x_3 - x_1)/(x_3 - x_2)}(\pm\Delta_{3,1})^{-(x_2 - x_1)/(x_3 - x_2)}, \qquad d_n = \pm(\pm\Delta_{n-1})^{(x_n - x_{n-2})/(x_{n-1} - x_{n-2})}(\pm\Delta_{n,n-2})^{-(x_n - x_{n-1})/(x_{n-1} - x_{n-2})},$$
$$\Delta_{3,1} = \frac{f_3 - f_1}{x_3 - x_1}, \qquad \Delta_{n,n-2} = \frac{f_n - f_{n-2}}{x_n - x_{n-2}},$$
где знак $\pm1$ соответствует направлению монотонности данных ($+1$ — возрастание, $-1$ — убывание).

Построенный интерполянт является непрерывным вместе с первой производной, то есть принадлежит классу $C^1$. Значения соседних сегментов и их первые производные совпадают в исходных узлах.

Совпадение вторых производных соседних сегментов не требуется, поэтому вторая производная $\frac{\partial^2 C}{\partial K^2}$ в узлах может иметь разные левое и правое значения. Следовательно, рациональный сплайн хорошо подходит для сохранения формы улыбки и проверки статического арбитража, но его вторую производную нельзя считать достаточно гладкой для непосредственного вычисления *local volatility* по формуле Дюпира.

Далее пройдемся по всем улыбкам и для каждой построим свой рациональный сплайн.

При фиксированном сроке $T$:

1) Сортируем исходные страйки: $K_1<K_2<...<K_n$.

2) Берём форвард $F(T)$ и коэффициент дисконтирования $D(T)$, соответствующие рассматриваемому сроку.

3) Переходим к *forward moneyness* $x=\frac{K}{F(T)}$. Так как $K=xF(T)$, то $\frac{\partial C}{\partial x}=F(T)\frac{\partial C}{\partial K}$.

Условие отсутствия *vertical-spread-arbitrage* имеет вид $-D(T)\leq\frac{\partial C}{\partial K}\leq0$. В координате $x$ оно принимает вид $-D(T)F(T)\leq\frac{\partial C}{\partial x}\leq0$.

4) Оцениваем узловые производные $d_i=\left.\frac{\partial C_T(x)}{\partial x}\right|_{x=x_i}$ и ограничиваем их допустимым диапазоном $-D(T)F(T)\leq d_i\leq0$.

5) Для каждого интервала $[x_i,x_{i+1}]$ вычисляем параметр $r_i$ и строим интерполянт $C_T(x)$, удовлетворяющий условиям $C_T(x_i)=C_i$ и $C_T'(x_i)=d_i$.

6) Строим плотную равномерную сетку между минимальным и максимальным значениями *moneyness*. Все внутренние исходные узлы специально в сетку не добавляем, поскольку рациональный сплайн является только $C^1$-гладким и в узле его вторая производная может иметь разные значения слева и справа.

7) Для каждой точки плотной сетки выполняем обратное преобразование $K=xF(T)$ и вычисляем интерполированную call-цену $C_T(x)$.

8) Вычисляем производные по страйку: $\frac{\partial C}{\partial K}=\frac{1}{F(T)}\frac{\partial C}{\partial x}$ и $\frac{\partial^2C}{\partial K^2}=\frac{1}{F^2(T)}\frac{\partial^2C}{\partial x^2}$.

9) Проверяем отсутствие *vertical-spread-arbitrage* по условию $-D(T)\leq\frac{\partial C}{\partial K}\leq0$ и отсутствие *butterfly-arbitrage* по условию $\frac{\partial^2C}{\partial K^2}\geq0$.

10) Для устойчивого обращения формулы Блека переходим к OTM-ценам. При $K\geq F(T)$ используем call-цену, а при $K<F(T)$ вычисляем put-цену из паритета $P=C-D(T)(F(T)-K)$.

11) По полученной OTM-цене обращаем формулу Блека и находим *implied volatility*.

12) Для каждой точки сохраняем $T$, $K$, $x$, $\log(x)$, call-цену, нормированную call-цену, *implied volatility*, обе производные и результаты арбитражных проверок в общую таблицу.

In [3]:
interpolated_smiles = []
smile_splines = {}

arbitrage_cases = []
tolerance = 1e-8


for expiration, smile in surface.groupby('expiration'):
    smile = smile.sort_values('strike').copy()

    strike = smile['strike'].to_numpy()
    call_midf = smile['call_midf'].to_numpy()

    forward = smile['forward'].iloc[0]
    T = smile['T'].iloc[0]
    discount_factor = smile['discount_factor'].iloc[0]

    moneyness = strike / forward

    calls_slopes = np.clip(
        slopes_estimation(moneyness, call_midf),
        -discount_factor * forward,
        0
    )
    r = r_boundary(calls_slopes, call_midf, moneyness)

    interpolated_calls_rs = RationalCubicSplineInterpolator(
        x=moneyness, y=call_midf, r=r, d=calls_slopes
    )

    smile_splines[T] = {
        'spline': interpolated_calls_rs,
        'forward': forward,
        'discount_factor': discount_factor,
        'x_min': moneyness.min(),
        'x_max': moneyness.max()
    }

    moneyness_grid = np.linspace(moneyness.min(), moneyness.max(), 1000)
    #moneyness_grid = np.unique(np.append(moneyness_grid, moneyness))

    strike_grid = moneyness_grid * forward
    interpolated_call_prices = interpolated_calls_rs(moneyness_grid)

    first_derivative = interpolated_calls_rs.derivative(moneyness_grid, nu=1).reshape(-1) / forward
    second_derivative = interpolated_calls_rs.derivative(moneyness_grid, nu=2).reshape(-1) / forward**2

    vertical_spread_arbitrage = (
        (first_derivative > tolerance)
        | (first_derivative < -discount_factor - tolerance)
    )
    butterfly_arbitrage = second_derivative < -tolerance

    if vertical_spread_arbitrage.any() or butterfly_arbitrage.any():
        arbitrage_cases.append({
            'expiration': expiration,
            'vertical_spread_violations': vertical_spread_arbitrage.sum(),
            'butterfly_violations': butterfly_arbitrage.sum()
        })

    interpolated_option_prices = np.where(
        strike_grid >= forward,
        interpolated_call_prices,
        interpolated_call_prices - discount_factor * (forward - strike_grid)
    )

    interpolated_impl_vol = implied_vol(
        forward, T, strike_grid, interpolated_option_prices,
        strike_grid >= forward, discount_factor
    )

    interpolated_smiles.append(pd.DataFrame({
        'expiration': expiration,
        'T': T,
        'strike': strike_grid,
        'forward': forward,
        'discount_factor': discount_factor,
        'moneyness': moneyness_grid,
        'log_moneyness': np.log(moneyness_grid),
        'call_price': interpolated_call_prices,
        'normalized_call_price': interpolated_call_prices / (discount_factor * forward),
        'impl_vol': interpolated_impl_vol,
        'first_derivative': first_derivative,
        'second_derivative': second_derivative,
        'vertical_spread_arbitrage': vertical_spread_arbitrage,
        'butterfly_arbitrage': butterfly_arbitrage
    }))



interpolated_surface = pd.concat(interpolated_smiles, ignore_index=True)
arbitrage_cases = pd.DataFrame(arbitrage_cases)

arbitrage_cases

""


### Визуализация интерполированных улыбок

На графике отображаются отдельные точки *implied volatility*, полученные после интерполяции call-цен по страйку для каждого исходного срока $T_i$.

In [4]:
points = interpolated_surface[
    (interpolated_surface['T'] >= 7 / 365) &
    (interpolated_surface['T'] <= 1) &
    (interpolated_surface['log_moneyness'].abs() <= 0.3)
].dropna(subset=['impl_vol'])

plot_step = max(1, len(points) // 60_000)
plot_points = points.iloc[::plot_step]

fig = dark_3d_scatter(
    x=plot_points['log_moneyness'],
    y=plot_points['T'],
    z=plot_points['impl_vol'] * 100,
    title='SPX interpolated implied volatility smiles',
    zaxis_title='Implied volatility, %',
    name='Interpolated smiles',
    colorscale='Viridis'
)

fig.show(config={'displaylogo': False, 'responsive': True})

## Интерполяция по $T$: ##

Подготовка общей сетки:

1) Выбираем сплайны со сроками $\frac{7}{365}\leq T_i\leq1$ и сортируем их по возрастанию $T_i$.

2) Для каждой улыбки переходим к *log-moneyness* $k=\log\frac{K}{F(T_i)}$ и ищем общую область $[k_{\min},k_{\max}]$, на которой определены все сплайны. Использование пересечения областей позволяет не применять экстраполяцию по страйку.

3) На общей области строим равномерную сетку из 201 точки: $k_1,...,k_{201}$.

4) Возвращаемся к *moneyness* по формуле $x_j=e^{k_j}$.

5) Для каждого срока $T_i$ вычисляем call-цену в страйке $K_{ij}=F(T_i)x_j$.

6) Нормируем call-цену по формуле $c(T_i,x_j)=\frac{C(T_i,F(T_i)x_j)}{D(T_i)F(T_i)}$.

В результате получаем матрицу узловых нормированных цен $\left[c(T_i,x_j)\right]$, строки которой соответствуют исходным срокам $T_i$, а столбцы — общей сетке *moneyness* $x_j$.

In [5]:
slice_items = sorted(
    (T, info) for T, info in smile_splines.items()
    if 7 / 365 <= T <= 1
)

T_nodes = np.array([T for T, _ in slice_items])

k_min = max(-0.3, max(np.log(info['x_min']) for _, info in slice_items))
k_max = min(0.3, min(np.log(info['x_max']) for _, info in slice_items))

log_moneyness_grid = np.linspace(k_min, k_max, 1000)
moneyness_grid = np.exp(log_moneyness_grid)

normalized_call_nodes = []

for T, info in slice_items:
    call_prices = info['spline'](moneyness_grid).reshape(-1)
    normalized_call_prices = call_prices / (info['discount_factor'] * info['forward'])
    normalized_call_nodes.append(normalized_call_prices)

normalized_call_nodes = np.asarray(normalized_call_nodes)

T_grid = np.unique(np.append(
    np.linspace(T_nodes.min(), T_nodes.max(), 1000),
    T_nodes
))

print('Область по log(K/F):', k_min, k_max)
print('Область по K/F:', np.exp(k_min), np.exp(k_max))
print('Количество исходных сроков:', len(T_nodes))
print('Количество интерполированных сроков:', len(T_grid))
print('Календарных нарушений в исходных срезах:', np.sum(np.diff(normalized_call_nodes, axis=0) < -1e-10))

Область по log(K/F): -0.21578084489479002 0.03598151172507225
Область по K/F: 0.8059119022290611 1.0366366806898002
Количество исходных сроков: 35
Количество интерполированных сроков: 1033
Календарных нарушений в исходных срезах: 0


In [6]:
def build_time_surface(normalized_calls):
    T_mesh, x_mesh = np.meshgrid(T_grid, moneyness_grid, indexing='ij')

    option_prices = np.where(
        x_mesh >= 1,
        normalized_calls,
        normalized_calls - (1 - x_mesh)
    )

    impl_vol = implied_vol(
        np.ones_like(x_mesh),
        T_mesh,
        x_mesh,
        option_prices,
        x_mesh >= 1,
        np.ones_like(x_mesh)
    )

    return pd.DataFrame({
        'T': T_mesh.reshape(-1),
        'moneyness': x_mesh.reshape(-1),
        'log_moneyness': np.log(x_mesh).reshape(-1),
        'normalized_call_price': normalized_calls.reshape(-1),
        'impl_vol': impl_vol.reshape(-1),
        'total_variance': (impl_vol**2 * T_mesh).reshape(-1),
    })

### Линейная интерполяция нормированных цен по времени

Для каждого фиксированного значения *moneyness* $x_j$ отдельно интерполируем зависимость $T\mapsto c(T,x_j)$.

Если $T_i\leq T\leq T_{i+1}$, вводим вес $\lambda=\frac{T-T_i}{T_{i+1}-T_i}$ и определяем цену как $c(T,x_j)=(1-\lambda)c(T_i,x_j)+\lambda c(T_{i+1},x_j)$.

Таким образом, между двумя соседними исходными сроками нормированная call-цена является линейной функцией времени.

При фиксированном $x_j$ линейная комбинация двух соседних ценовых срезов сохраняет границы нормированной call-цены, убывание по *moneyness* и выпуклость по *moneyness*. Неубывание по времени также сохраняется, если в исходных узлах выполняется условие $c(T_{i+1},x_j)\geq c(T_i,x_j)$.

Поэтому линейную интерполяцию нормированных цен используем как простой безарбитражный ценовой *baseline*.

In [7]:
linear_normalized_calls = np.column_stack([
    np.interp(T_grid, T_nodes, normalized_call_nodes[:, j])
    for j in range(len(moneyness_grid))
])

linear_time_surface = build_time_surface(linear_normalized_calls)

linear_time_surface

,T,moneyness,log_moneyness,normalized_call_price,impl_vol,total_variance
0,0.019178,0.805912,-0.215781,0.194147,0.562593,0.006070
1,0.019178,0.806115,-0.215529,0.193944,0.562167,0.006061
2,0.019178,0.806318,-0.215277,0.193741,0.561741,0.006052
3,0.019178,0.806521,-0.215025,0.193538,0.561313,0.006042
4,0.019178,0.806725,-0.214773,0.193335,0.560885,0.006033
...,...,...,...,...,...,...
1032995,0.904110,1.035592,0.034973,0.042226,0.151153,0.020656
1032996,0.904110,1.035853,0.035225,0.042097,0.151069,0.020634
1032997,0.904110,1.036114,0.035477,0.041971,0.150996,0.020614
1032998,0.904110,1.036375,0.035729,0.041847,0.150925,0.020594


### Проверка линейной ценовой поверхности

Проверяем полученную поверхность нормированных call-цен на общей сетке.

1) Границы цены. Для каждой точки должны выполняться условия $(1-x_j)^+\leq c(T_i,x_j)\leq1$.

2) *Calendar arbitrage*. При фиксированном *forward moneyness* цена не должна убывать с увеличением срока: $c(T_{i+1},x_j)-c(T_i,x_j)\geq0$.

3) *Vertical-spread-arbitrage*. На каждом интервале вводим дискретный наклон $s_j(T_i)=\frac{c(T_i,x_{j+1})-c(T_i,x_j)}{x_{j+1}-x_j}$. Для отсутствия арбитража требуется $-1\leq s_j(T_i)\leq0$.

4) *Butterfly-arbitrage*. Дискретные наклоны должны быть неубывающими: $s_{j+1}(T_i)-s_j(T_i)\geq0$. Это является дискретным аналогом условия выпуклости $\frac{\partial^2c}{\partial x^2}\geq0$.

Все условия проверяются численно с учётом небольшой погрешности `tolerance`.

In [8]:
linear_surface_report = surface_arbitrage_report(
    linear_normalized_calls,
    moneyness_grid
)

Нарушения нижней границы цены: 0
Нарушения верхней границы цены: 0
Нарушения calendar arbitrage: 0
Нарушения верхней границы наклона: 0
Нарушения нижней границы наклона: 0
Нарушения butterfly arbitrage: 0
Диапазон наклона: -0.9990813496836712 -0.0043009893189998975


### Поверхность implied volatility, соответствующая линейной интерполяции цен

После интерполяции нормированных call-цен переходим к OTM-опционам и в каждой точке обращаем формулу Блека.

Полученная *implied volatility* не интерполируется непосредственно. Она восстанавливается из линейно интерполированных опционных цен.

Чёрными линиями на графике отмечаются исходные сроки $T_i$. Остальные точки соответствуют промежуточным срокам, для которых сначала была найдена нормированная опционная цена, а затем вычислена *implied volatility*.

In [9]:
T_mesh, k_mesh = np.meshgrid(T_grid, log_moneyness_grid, indexing='ij')
impl_vol_mesh = linear_time_surface['impl_vol'].to_numpy().reshape(len(T_grid), len(log_moneyness_grid))

time_step = max(1, len(T_grid) // 300)
k_step = max(1, len(log_moneyness_grid) // 160)

fig = dark_3d_surface(
    x=k_mesh[::time_step, ::k_step],
    y=T_mesh[::time_step, ::k_step],
    z=impl_vol_mesh[::time_step, ::k_step] * 100,
    title='Linear interpolation of normalized call prices in time',
    zaxis_title='Implied volatility, %',
    name='Implied volatility',
    colorscale='Viridis'
)

node_rows = np.searchsorted(T_grid, T_nodes)

for index, (row, T) in enumerate(zip(node_rows, T_nodes)):
    fig.add_trace(go.Scatter3d(
        x=log_moneyness_grid,
        y=np.full_like(log_moneyness_grid, T),
        z=impl_vol_mesh[row] * 100,
        mode='lines',
        name='Исходные экспирации',
        legendgroup='node_expirations',
        showlegend=index == 0,
        line=dict(color='#f0f6fc', width=2),
        opacity=0.45,
        hovertemplate='k: %{x:.3f}<br>T: %{y:.3f}<br>IV: %{z:.2f}%<extra></extra>'
    ))

fig.show(config={'displaylogo': False, 'responsive': True})

### Почему линейная интерполяция цен не используется непосредственно для вычисления local volatility

На предыдущем этапе линейно интерполировалась не *local volatility*, а нормированная call-цена $c(T,x)=\frac{C(T,F(T)x)}{D(T)F(T)}$, где $x=\frac{K}{F(T)}$.

Линейная интерполяция нормированных цен сохраняет основные условия отсутствия статического арбитража, поэтому полученная поверхность остаётся полезным безарбитражным ценовым *baseline*.

Однако внутри каждого интервала $[T_i,T_{i+1}]$ временная производная постоянна и равна $\frac{\partial c}{\partial T}(T,x)=\frac{c(T_{i+1},x)-c(T_i,x)}{T_{i+1}-T_i}$.

При переходе через исходный срок $T_i$ эта производная скачкообразно меняется, поскольку её значения на интервалах $[T_{i-1},T_i]$ и $[T_i,T_{i+1}]$ в общем случае не совпадают. Следовательно, линейная ценовая поверхность непрерывна по времени, но не является гладкой по времени.

Формула Дюпира использует производную по времени и вторую производную по страйковой координате. Поэтому её непосредственное применение к линейной ценовой поверхности приводит к разрывной по времени *local volatility*, а в исходных сроках единая временная производная вообще не определена.

По этой причине линейную интерполяцию нормированных цен сохраняем для ценового *baseline*, но не дифференцируем её непосредственно для вычисления гладкой *local volatility*.

Далее по узловым ценам восстанавливаем *implied volatility*, вычисляем полную подразумеваемую дисперсию $w(T,k)=T\sigma_{\mathrm{impl}}^2(T,k)$, где $k=\log\frac{K}{F(T)}$, и строим отдельную гладкую поверхность $\widehat{w}(T,k)$.

Сглаженная поверхность $\widehat{w}(T,k)$ является новым интерполянтом. Отсутствие арбитража у линейной ценовой поверхности не гарантирует автоматически отсутствие арбитража у $\widehat{w}(T,k)$, поэтому после сглаживания необходимо повторно проверить условия *calendar-arbitrage* и *butterfly-arbitrage*.

In [10]:
T_save_mesh, k_save_mesh = np.meshgrid(T_nodes, log_moneyness_grid, indexing='ij')

rational_price_nodes = pd.DataFrame({
    'T': T_save_mesh.ravel(),
    'log_moneyness': k_save_mesh.ravel(),
    'moneyness': np.exp(k_save_mesh).ravel(),
    'normalized_call_price': normalized_call_nodes.ravel()
})

rational_price_nodes.to_parquet(PROCESSED_DIR / 'rational_price_nodes.parquet', index=False)

print('Сохранено строк:', len(rational_price_nodes))
print('Количество сроков:', rational_price_nodes['T'].nunique())
print('Количество значений log-moneyness:', rational_price_nodes['log_moneyness'].nunique())

Сохранено строк: 35000
Количество сроков: 35
Количество значений log-moneyness: 1000
